In [ ]:
from theia.terrain import SrtmTerrainModel


terrain = SrtmTerrainModel()

In [ ]:
from theia.types import Point


THEATER_LAT_MIN = 46.3805
THEATER_LON_MIN = 9.2207
THEATER_LAT_MAX = 47.2440
THEATER_LON_MAX = 10.4790

# Source:
# AIRAC AIP SUP: 008/2025
# https://www.skybriefing.com/documents/10156/531923/LS_Sup_A_2025_008_en.pdf/9383f427-aee4-73e5-2f58-f6c49c721bc8?t=1766406342056
# Retrieved 2026-07-02.

NM_TO_M = 1852
FEET_TO_M = 0.3048

DAVOS_LAT = 46.81472
DAVOS_LON = 9.84944
RADIUS_RESTRICTED_AREA = 25 * NM_TO_M  # [m]
RADIUS_DAVOS_CONTROL_ZONE = 2.7 * NM_TO_M  # [m]
HEIGHT_RESTRICTED_AREA = 19500 * FEET_TO_M  # [m]

davos = Point(
    lat=DAVOS_LAT,
    lon=DAVOS_LON,
    alt=terrain.elevationAt(DAVOS_LAT, DAVOS_LON),
)

In [ ]:
import json
import folium

data_dir = "/home/user/Documents/theia_backend/examples/conference/data"

with open(
    f"{data_dir}/LS-R90.geojson", "r"
) as file:
    lsr90 = json.load(file)

with open(f"{data_dir}/flight-path-spacetime_authorized.json", "r") as file:
    data_authorized = json.load(file)

with open(f"{data_dir}/flight-path-spacetime_adversarial.json", "r") as file:
    data_adversarial = json.load(file)



flight_path_green = folium.PolyLine(
    [(p["lat"], p["lon"]) for p in data_authorized if p["time"] <= 1550],
    tooltip="Normal flight",
    color="green",
)
flight_path_green_planned = folium.PolyLine(
    [(p["lat"], p["lon"]) for p in data_authorized if p["time"] > 1550],
    tooltip="Normal flight (authorized)",
    color="green",
    dashArray="5, 5",
)
flight_path_red = folium.PolyLine(
    [(p["lat"], p["lon"]) for p in data_adversarial if p["time"] > 1550],
    tooltip="Unauthorized change of plan: RED!",
    color="red",
)

In [ ]:
import folium


highlight_waypoints = [
    "AKABI",
    "BODAN",
    "LAGOS",
    "VEBEG",
    "EBUXA",
    "ARGAX",
]


map = folium.Map(location=(DAVOS_LAT, DAVOS_LON), zoom_start=9)
folium.LatLngPopup().add_to(map)
folium.Rectangle(
    [
        (THEATER_LAT_MIN, THEATER_LON_MIN),
        (THEATER_LAT_MAX, THEATER_LON_MAX),
    ],
    tooltip="Theater",
    color="black",
).add_to(map)

folium.GeoJson(
    lsr90,
    tooltip="TEMPO LS-R90",
    fill=False,
    color="orange",
).add_to(map)
folium.Circle(
    location=(DAVOS_LAT, DAVOS_LON),
    radius=RADIUS_DAVOS_CONTROL_ZONE,
    tooltip="CTR",
    color="darkred",
).add_to(map)

folium.Marker(
    flight_path_green.locations[-1],
    tooltip="Transponder off",
    icon=folium.Icon(icon="power-off", prefix="fa", color="red"),
).add_to(map)
flight_path_green.add_to(map)
flight_path_green_planned.add_to(map)
flight_path_red.add_to(map)

map

In [ ]:
map.save("map.html")

In [ ]:
# import pandas as pd


# waypoints = pd.read_csv("data/opennav_waypoints_switzerland.csv")

In [ ]:
# import numpy as np

# from theia.coordinates import CoordinateTransformations, EcefToEnuTransformer


# lat_lons = (
#     waypoints.set_index("IDENT")
#     .loc[
#         [
#             "AKABI",
#             "BODAN",
#             "LAGOS",
#             "VEBEG",
#             "EBUXA",
#             "ARGAX",
#             # "PERER",
#             # "GUGSA",
#         ],
#         ["lat", "lon"],
#     ]
#     .values
# )

# alt = 1000.0

# transformer = EcefToEnuTransformer(
#     Point(
#         lat=0.5 * (THEATER_LAT_MIN + THEATER_LAT_MAX),
#         lon=0.5 * (THEATER_LON_MIN + THEATER_LON_MAX),
#         alt=alt,
#     )
# )

# p_enu = np.array(
#     [
#         transformer.ecef_to_enu(
#             CoordinateTransformations.geodetic_to_cartesian(lat, lon, alt)
#         )
#         for lat, lon in lat_lons
#     ]
# )
# p_enu

In [ ]:
# import numpy as np
# from scipy.optimize import least_squares

# from theia.maneuvers import ConstantSpeedCurveManeuver


# # def build_curve_from_maneuver(points: list[Point]) -> ConstantSpeedCurveManeuver:
# def fit_arc(points, min_radius, z_reg=5.0):
#     """
#     points: (N,3) ENU. Design space: center (cx,cy,cz), radius r -> dim 4.
#     z_reg pulls center altitude toward the data's mean altitude; without it,
#     a sphere fit to a shallow/near-planar arc is ill-conditioned (many
#     tilted spheres explain a thin slice almost equally well).
#     """
#     mean_z = points[:, 2].mean()

#     def residuals(x):
#         center, r = x[:3], x[3]
#         geom = np.linalg.norm(points - center, axis=1) - r
#         return np.append(geom, z_reg * (center[2] - mean_z))

#     c0 = points.mean(axis=0)
#     r0 = max(np.linalg.norm(points - c0, axis=1).mean(), min_radius)
#     x0 = np.append(c0, r0)

#     res = least_squares(
#         residuals, x0, bounds=([-np.inf] * 3 + [min_radius], [np.inf] * 4)
#     )
#     center, radius = res.x[:3], res.x[3]
#     return center, radius, res


# v_max = 70.0
# max_turn_rate_deg_s = 3.0

# min_radius = v_max / np.deg2rad(max_turn_rate_deg_s)  # R = v / ω
# center, radius, res = fit_arc(p_enu, min_radius)
# p_center = CoordinateTransformations.cartesian_to_geodetic(
#     *transformer.enu_to_ecef(center)
# )

# # ConstantSpeedCurveManeuver(p_center, speed=v_max,)

In [ ]:
# import plotly.express as px
# import plotly.graph_objects as go

# fig = px.scatter_3d(
#     x=p_enu[:, 0],
#     y=p_enu[:, 1],
#     z=p_enu[:, 2],
#     labels={"x": "East [m]", "y": "North [m]", "z": "Up [m]"},
# )
# fig.add_trace(go.Scatter3d(x=[center[0]], y=[center[1]], z=[center[2]]))

# fig

In [ ]:
# Idea:
# An aircraft comes from Bodensee using waypoints AKABI - BODAN - LAGOS - VEBEG - EBUXA - ARGAX - PERER - GUGSA - Airport Samedan.

In [ ]:
# import re

# import pandas as pd


# df = pd.read_csv("data/opennav_waypoints_switzerland.csv").set_index("IDENT")


# def dms_to_decimal(degrees, minutes, seconds, direction):
#     """Convert DMS (degrees, minutes, seconds) to decimal degrees."""
#     decimal = degrees + minutes / 60 + seconds / 3600
#     if direction in ("S", "W"):
#         decimal = -decimal
#     return decimal


# def parse_dms_string(coord_str):
#     """Parse a DMS string like '47° 20' 48.00\" N' into decimal degrees."""
#     pattern = r"(\d+)[°\s]+(\d+)['\s]+([\d.]+)[\"\s]+([NSEW])"
#     match = re.search(pattern, coord_str)
#     if not match:
#         raise ValueError(f"Could not parse coordinate: {coord_str}")
#     degrees, minutes, seconds, direction = match.groups()
#     return dms_to_decimal(float(degrees), float(minutes), float(seconds), direction)


# pd.DataFrame(
#     [
#         {
#             "IDENT": ident,
#             "lat": parse_dms_string(row["LATITUDE"]),
#             "lon": parse_dms_string(row["LONGITUDE"]),
#         }
#         for ident, row in df.iterrows()
#     ]
# ).to_csv("data/opennav_waypoints_switzerland.csv")

from theia.data_loading import load_trajectory_file
from theia.distance import line_of_sight_distance
from theia.types import Trajectory


def is_within_distance(trajectory: Trajectory, point: Point, distance: float) -> bool:
    for lat, lon in zip(trajectory.lats, trajectory.lons, strict=True):
        d = line_of_sight_distance(
            lat,
            lon,
            point.alt,
            point.lat,
            point.lon,
            point.alt,
        )
        if d <= distance:
            return True
    return False


# trajectories, _ = load_trajectory_file("data/data_opensky_2022-06-27.csv")
# trajectories = [
#     t for t in trajectories if not is_within_distance(t, davos, RADIUS_RESTRICTED_AREA)
# ]

import pandas as pd

from theia.config import SIDC
from theia.types import ConstantRcsModel


df = pd.read_csv("data/data_opensky_2017-06-15.csv").drop(columns="callsign")
trajectories: list[Trajectory] = []
for i, (icao24, group) in enumerate(df.groupby("icao24")):
    if group.shape[0] < 2:
        continue
    t = Trajectory(
        target_id=i,
        target_sidc=SIDC.RED_FIXED_WING,
        times=group["time"],
        lats=group["lat"],
        lons=group["lon"],
        alts=group["alt"],
        vxs=[0.0 for _ in range(len(group))],
        vys=[0.0 for _ in range(len(group))],
        vzs=[0.0 for _ in range(len(group))],
        cross_section_model=ConstantRcsModel(rcs=1.0),
    )
    if max(t.alts) <= 3000:
        trajectories.append(t)

len(trajectories)

icaos = df["icao24"].unique()
df_aircraft_db = pd.read_csv("~/Downloads/aircraftDatabase.csv")
df_types_in_data = df_aircraft_db.query("`icao24`.isin(@icaos)")
df_types_in_data.to_csv("tmp.csv", index=False)

# df_types_in_data

# df_types_in_data["typecode"].str.contains("^L[12]P$").sum()

# for i in df["icao24"].unique():
#     print(i)

len(trajectories)